In [36]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [42]:
import os

from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemini-3.1-flash-lite",
    model_provider="google-genai",
    api_key=os.getenv("GOOGLE_API_KEY"),
)


In [43]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [44]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is inquiring about fictional details regarding the moon, specifically the city of "Lunapolis," including its environmental conditions, population demographics (cheese miners), and internal political/labor stability.\n\n## SUMMARY\n\n*   **Location:** The capital of the moon is identified as Lunapolis.\n*   **Environment:** The climate is extreme, with surface temperatures ranging from -100°C to 120°C.\n*   **Demographics:** The city has a population of 100,000 "cheese miners."\n*   **Political Status:** There is current labor unrest; a strike by the cheese miners\' union is anticipated due to dissatisfaction with the new president.\n\n## ARTIFACTS\n\nNone.\n\n## NEXT STEPS\n\nProvide further information regarding the political landscape of Lunapolis or answer additional inquiries about the moon\'s fictional infrastructure as requested by the user.', additional_kwargs={'lc

In [45]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user is inquiring about fictional details regarding the moon, specifically the city of "Lunapolis," including its environmental conditions, population demographics (cheese miners), and internal political/labor stability.

## SUMMARY

*   **Location:** The capital of the moon is identified as Lunapolis.
*   **Environment:** The climate is extreme, with surface temperatures ranging from -100°C to 120°C.
*   **Demographics:** The city has a population of 100,000 "cheese miners."
*   **Political Status:** There is current labor unrest; a strike by the cheese miners' union is anticipated due to dissatisfaction with the new president.

## ARTIFACTS

None.

## NEXT STEPS

Provide further information regarding the political landscape of Lunapolis or answer additional inquiries about the moon's fictional infrastructure as requested by the user.


## Trim/delete messages

In [46]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [ ]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [47]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is seeking assistance to troubleshoot a device (model: Blorp-X7) that fails to power on. The goal is to identify the cause of the failure and restore functionality.\n\n## SUMMARY\n\n- Device: Blorp-X7.\n- Current Status: User confirms the device is plugged into a power source and turned on.\n- Diagnostic Data: A ping diagnostic was initiated; readings show 42°C and 2.9v.\n- Current Step: Assessing external status indicators to determine if there is any electrical response or light activity.\n\n## ARTIFACTS\n\nNone.\n\n## NEXT STEPS\n\n- Evaluate the user's response regarding visual indicators (lights) to determine if the issue is a power delivery failure, a boot issue, or a hardware defect.\n- Proceed with further diagnostic steps or hardware inspection based on the presence or absence of power indicators.", additional_kwargs={'lc_source': 'summarization'}, response_metad

In [48]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'Based on the diagnostic data from the previous step, the device is currently reading **42°C**.', 'extras': {'signature': 'EnEKbwFpFH0T6yg68SLAV+kfISOXSqy5CingnGwKGNgvueau+axnKR7mpKXVT60uDn9HrP37wV+eUnkNnNaiDCFPfHID3+4Mm2ErpuRcWEUfdALrVLB8mOBLjF2LLriap+psoGTfZjy/g++PD/NxMEKsTw=='}}]
